# 06 — Relearn task 6–7, dữ liệu (a): cutout, bảng camera và độ phủ các mặt của 50 SKU

Ngày tạo: 31/10/2026. Nhánh `feat/task6-relearn`. Theo [plan](../../docs/project/task6-relearn-plan-2026-10-30.md), Task 9, quyết định Q5: mỗi SKU đủ các mặt RPC đã chụp, dùng góc thấy nhãn, phân bố cân đối.

**Làm gì:** kiểm kê hai bộ cutout mới của 50 SKU relearn; tóm tắt kết quả duyệt (`reviews.json`) và bảng camera (`skus.json`); chốt catalog cutout (M0) gồm 6 bộ v1 cộng 2 bộ mới; kiểm cổng độ phủ Q5 cho task 6–7.

**Không chạy lại trong notebook:** giải nén 1.985 ảnh cam2 còn thiếu từ zip Kaggle; cắt SAM 2.1 base trên GPU (`--per-class 200 --camera-weights 0:1,1:0,2:1,3:0`, log `runs/data_preprocessing/synthetic_relearn/extract.log`); cắt lại bằng SAM 2.1 large 1.086 ảnh cam0 bị SAM base loại vì iou < 0,9 (`resegment.py`, ngưỡng 0,75, log `recut_l.log`); duyệt bằng mắt trên trang lưới (`cutouts/`, `cutouts_l/`, danh sách loại `rejects_b.txt`, `rejects_l.txt`, ghép bằng `build_reviews.py`). Các bước đó chạy ngày 31/10, ở đây chỉ kiểm hash.

**Quyết định của người dùng (chat 31/10):** xác nhận gallery cutout bị loại; chốt bảng camera; chọn phương án A cho 7 SKU trượt cổng khe cam0 (ghi ngoại lệ có lý do trong `configs/data/synth_relearn.json`, giữ luật M0 loại góc hộp lộ hai mặt).

**Input:** `data/archive` (train2019, instances), `data/processed/cutouts_relearn_v1`, `data/processed/cutouts_relearn_l_v1`, catalog v1 `data/processed/rpc_synth_v2_v1/catalog`. **Output:** `data/processed/rpc_synth_relearn_v1/catalog/{cutouts_relearn_v1,cutouts_relearn_l_v1,cutouts,m0_review,coverage}.json`, run receipt ở `runs/data_preprocessing/synthetic_relearn/<run_id>/`.

In [1]:
import json, os, platform, subprocess, sys, uuid, hashlib, time
from collections import Counter
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pdp/main.py").exists())
os.chdir(REPO); sys.path.insert(0, str(REPO))
TZ = ZoneInfo("Asia/Bangkok")
RUN_ID = uuid.uuid4().hex[:8]
STARTED_AT = datetime.now(TZ).isoformat(timespec="seconds")
RUN_DIR = REPO / "runs/data_preprocessing/synthetic_relearn" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
WORK = REPO / "runs/data_preprocessing/synthetic_relearn"
V1 = Path("data/processed/rpc_synth_v2_v1/catalog")
CAT = Path("data/processed/rpc_synth_relearn_v1/catalog")
CONFIG = "configs/data/synth_relearn.json"
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
ENV = {"run_id": RUN_ID, "started_at": STARTED_AT, "git_revision": git("rev-parse", "HEAD"),
       "git_dirty": bool(git("status", "--porcelain", "--", "autocheckout", "tools", "tests")),
       "python": sys.version.split()[0], "host": platform.node()}
print(json.dumps(ENV, indent=1))

{
 "run_id": "ecab3dfc",
 "started_at": "2026-10-31T12:46:12+07:00",
 "git_revision": "abb86d883953079e339c8cb25bc710172533e4d8",
 "git_dirty": false,
 "python": "3.10.22",
 "host": "archlinux"
}


## 1. Hai bộ cutout mới và kiểm kê (Step 1–3)

`relearn_v1`: SAM 2.1 base, mọi ảnh cam0 và cam2 của 50 SKU (5.438 ảnh, loại tự động khi iou < 0,9 hoặc fill ngoài 0,3–0,98). `relearn_l_v1`: SAM 2.1 large cắt lại các ảnh cam0 bị loại vì iou. Gần hết các bước cam0 thiếu (1.036/1.038) là do loại iou chứ không do RPC thiếu ảnh; lô thử 60 ảnh cho thấy SAM-L không nâng iou dự đoán (trung vị 0,869) nhưng mask nhìn sạch, nên ngưỡng hạ xuống 0,75 và chất lượng do vòng duyệt bằng mắt quyết định. Kiểm kê: file tồn tại, alpha khác rỗng, ảnh nguồn thuộc train2019, SHA-256.

In [2]:
def run(*args):
    t = time.time()
    r = subprocess.run([sys.executable, "-m", "tools.synth_v2", *args], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-2000:]); assert r.returncode == 0, r.stderr[-2000:]
    return round(time.time() - t, 1)

SETS = {"relearn_v1": "data/processed/cutouts_relearn_v1", "relearn_l_v1": "data/processed/cutouts_relearn_l_v1"}
for name, d in SETS.items():
    run("inventory", "--index", f"{d}/index.json", "--cutout-set", name, "--root", "data/archive",
        "--out", str(CAT / f"cutouts_{name}.json"))
    assert not json.load(open(CAT / f"cutouts_{name}.json"))["issues"]
    rejected = json.load(open(f"{d}/rejected.json"))
    print(" ", name, "rejected at extraction:", dict(Counter(r["reason"] for r in rejected)))
for log in ("extract.log", "recut_l.log"):
    print(log, sha(WORK / log)[:16], (WORK / log).read_text().strip().splitlines()[-2])

3568 usable cutouts, 0 issues, cameras {0: 1632, 2: 1936} -> data/processed/rpc_synth_relearn_v1/catalog/cutouts_relearn_v1.json
  relearn_v1 rejected at extraction: {'iou': 1834, 'fill': 36}


1020 usable cutouts, 0 issues, cameras {0: 1020} -> data/processed/rpc_synth_relearn_v1/catalog/cutouts_relearn_l_v1.json
  relearn_l_v1 rejected at extraction: {'iou': 66}
extract.log 9081142a23fbfba2 CANH BAO: 37 lop co it hon 100 cutout: [33, 34, 35, 36, 73, 74, 79, 80, 82, 83, 90, 91, 92, 93, 97, 98, 102, 108, 109, 110]... -> xem rejected.json
recut_l.log a9df15f0eea88d50 1086 targets: 1020 ok, 66 rejected, 513s


## 2. Kết quả duyệt (Step 4)

`reviews.json` = 6.625 review của v1 + review của cutout mới. Cutout mới có ảnh nguồn mà v1 đã giữ (`keep`) bị loại theo luật (`duplicate_source`, mỗi annotation chỉ giữ một cutout). Phần còn lại duyệt bằng mắt theo tiêu chí M0: đĩa xoay hoặc mảnh đĩa trong mask, mask rách, sai vật, góc hộp lộ hai mặt (≥ ~30% bề rộng). 511 cutout SAM-L cam0 của 15 SKU chỉ dùng cam2 không được duyệt: generator không dùng chúng và `finalize` không giữ cutout chưa duyệt. Người dùng xác nhận gallery cutout bị loại (`reject_gallery/`) ngày 31/10.

In [3]:
reviews = json.load(open(CAT / "reviews.json"))
new_ids = {c["cutout_id"] for n in SETS for c in json.load(open(CAT / f"cutouts_{n}.json"))["cutouts"]}
new = [r for r in reviews["reviews"] if r["cutout_id"] in new_ids]
print("reviews", len(reviews["reviews"]), "| new", len(new), "| unreviewed new", len(new_ids) - len(new))
print(Counter((r["reviewer"], r["decision"]) for r in new))
print(Counter(r["reason"] for r in new if r["decision"] == "reject"))
print(reviews["status"])
for p in ("rejects_b.txt", "rejects_l.txt", "build_reviews.py"):
    print(p, sha(WORK / p)[:16])
print("reviews.json", sha(CAT / "reviews.json")[:16])

reviews 10702 | new 4077 | unreviewed new 511
Counter({('claude', 'keep'): 2216, ('rule', 'reject'): 1647, ('claude', 'reject'): 214})
Counter({'duplicate_source': 1647, 'box view shows two faces': 140, 'turntable disc in mask': 32, 'small turntable fragment': 24, 'broken/torn mask': 16, 'wrong object / broken mask': 2})
relearn review done 31/10; reject gallery confirmed by the user (chat 31/10/2026)
rejects_b.txt 934d7ce392e382cb
rejects_l.txt 77ab56af44c4dec0
build_reviews.py fc5270e15a2f992f
reviews.json 264847cfbbd4d6c9


## 3. Bảng camera (Step 5)

Bản sao `skus.json` v1, xét lại 50 SKU trên trang `sheets/` (ảnh val | cam2 | cam0) theo Q5(b): góc nào thấy nhãn thì dùng. Người dùng chốt ngày 31/10: chỉ 119 (flat → mixed: val nửa ngửa nắp, nửa nằm nghiêng lộ nhãn) và 121 (mixed 0,3/0,7 → upright: val gần như toàn nằm nghiêng, nay có 40 cutout cam0) đổi; 123/130 giữ flat (val chỉ thấy mặt trên có nhãn, cam0 là cạnh mỏng), 134/135 giữ mixed.

In [4]:
from autocheckout.taskcfg import TaskConfig

old = {s["rpc_category_id"]: s for s in json.load(open(V1 / "skus.json"))["skus"]}
skus = json.load(open(CAT / "skus.json"))["skus"]
changed = [s for s in skus if s["camera_weights"] != old[s["rpc_category_id"]]["camera_weights"]]
for s in changed:
    o = old[s["rpc_category_id"]]
    print(s["name"], o["shape_group"], o["camera_weights"], "->", s["shape_group"], s["camera_weights"], "|", s["accepted_by"])
task_config = TaskConfig.load(json.load(open(CONFIG))["task_config"])
relearn = {c.rpc_category_id for t in (6, 7) for c in task_config.task(t).classes}
print("shape groups of the 50:", Counter(s["shape_group"] for s in skus if s["rpc_category_id"] in relearn))
print("skus.json", sha(CAT / "skus.json")[:16])

119_canned_food flat {'2': 1.0} -> mixed {'0': 0.5, '2': 0.5} | user 31/10 (AskUserQuestion)
121_canned_food mixed {'0': 0.3, '2': 0.7} -> upright {'0': 0.99, '2': 0.01} | user 31/10 (AskUserQuestion)
shape groups of the 50: Counter({'upright': 27, 'flat': 17, 'mixed': 6})
skus.json 4a1bd9551b5405d5


## 4. Chốt M0 và cổng độ phủ Q5 (Step 6)

Catalog cuối = cutout `keep` trên camera mà SKU dùng, từ 6 bộ v1 và 2 bộ mới. Cổng Q5: mọi kiểu đặt × camera có cutout, và khe xoay cam0 lớn nhất ≤ 2 bước (18°). 7 SKU trượt được người dùng chấp nhận làm ngoại lệ (phương án A, lý do ghi trong config): 38, 97, 98, 102 là hộp, khe nằm ở các góc vì luật M0 loại góc lộ hai mặt nhưng cả bốn mặt đều có ảnh; 90 và 198 khe 27° do mask rách; 112 do RPC không có ảnh cam0 bước 1–2.

In [5]:
v1_sets = ["sam_v1", "cam0_v1", "cam2s_v1", "cam2s_v2", "cam0s_v2", "repl_v1"]
catalogs = [str(V1 / f"cutouts_{n}.json") for n in v1_sets] + [str(CAT / f"cutouts_{n}.json") for n in SETS]
run("finalize", "--catalogs", *catalogs, "--reviews", str(CAT / "reviews.json"), "--skus", str(CAT / "skus.json"),
    "--out-dir", str(CAT), "--accepted-by", "user (chat 31/10/2026)",
    "--notes", "relearn: v1 catalog + SAM-base and SAM-L camera-0 cuts of the 50 SKUs; reject gallery and camera table "
               "confirmed by the user 31/10; coverage exceptions 38 90 97 98 102 112 198 accepted (option A)")
final = json.load(open(CAT / "cutouts.json"))["cutouts"]
per_sku = Counter(c["rpc_category_id"] for c in final)
print("final cutouts", len(final), "by set", dict(Counter(c["cutout_set"] for c in final)))
print("min per SKU: all", min(per_sku.values()), "| the 50", min(per_sku[r] for r in relearn))
run("coverage", "--config", CONFIG, "--tasks", "6", "7")
cov = json.load(open(CAT / "coverage.json"))
for rpc, reason in cov["exceptions"].items():
    print(rpc, cov["skus"][rpc]["cam0_gap"], "-", reason)

M0: 7406 cutouts, min 15/SKU -> data/processed/rpc_synth_relearn_v1/catalog
final cutouts 7406 by set {'cam0_v1': 1369, 'cam0s_v2': 2, 'cam2s_v1': 118, 'cam2s_v2': 926, 'relearn_l_v1': 445, 'relearn_v1': 1084, 'repl_v1': 184, 'sam_v1': 3278}
min per SKU: all 15 | the 50 40


43/50 SKUs cover every placement, camera and angle, 7 excepted -> data/processed/rpc_synth_relearn_v1/catalog/coverage.json
38 {'back': 9, 'front': 7} - box: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
90 {'front': 3} - flat glass bottle: torn camera-0 masks rejected, widest gap 3 steps (27 degrees)
97 {'front': 5} - milk carton: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
98 {'front': 3} - milk carton: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
102 {'front': 3} - milk carton: camera-0 corner views showing two faces rejected (M0), all four faces kept; gaps at the corners
112 {'front': 3} - RPC train2019 has no camera-0 photo at steps 1-2 (gap 3 steps)
198 {'front': 3} - glue stick: torn camera-0 masks rejected, widest gap 3 steps (27 degrees)


In [6]:
FINISHED_AT = datetime.now(TZ).isoformat(timespec="seconds")
summary = {**ENV, "finished_at": FINISHED_AT, "status": "PASS_M0_Q5",
           "outputs": {p: sha(CAT / p) for p in ("cutouts.json", "m0_review.json", "coverage.json", "skus.json", "reviews.json",
                                                 "cutouts_relearn_v1.json", "cutouts_relearn_l_v1.json")},
           "config_sha256": sha(CONFIG),
           "limits": ["visual review by one reviewer (Claude) with user confirmation of the reject gallery",
                      "camera-0 SAM-L cutouts kept from predicted iou 0.75, judged by eye only",
                      "7 SKUs pass the Q5 gate as user-accepted exceptions",
                      "SAM extraction, the Kaggle unzip and the visual review are not re-run here"]}
json.dump(summary, open(RUN_DIR / "summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

{
 "run_id": "ecab3dfc",
 "started_at": "2026-10-31T12:46:12+07:00",
 "git_revision": "abb86d883953079e339c8cb25bc710172533e4d8",
 "git_dirty": false,
 "python": "3.10.22",
 "host": "archlinux",
 "finished_at": "2026-10-31T12:46:33+07:00",
 "status": "PASS_M0_Q5",
 "outputs": {
  "cutouts.json": "5d9335188c31bb06f1befc3e0630b66194b448e1be9eedba9670b49f882b267f",
  "m0_review.json": "18ece652dd7b3652af32d9a2657559ca2d08bf47822294196c6a2ad80f1992bf",
  "coverage.json": "5da9ae050aae799e46e46e22c60b2f03500ceba2d70115e1a00e88f3ac513731",
  "skus.json": "4a1bd9551b5405d5eb4a594497abe0d250d2ea2131ae8fb885470e9d17d502c3",
  "reviews.json": "264847cfbbd4d6c9dd609483a67bf880934b3ab1f017ba150397476a8d555632",
  "cutouts_relearn_v1.json": "aaa43de1ee063109d961ff2ebcbed4905344b6d7a3759a078314d779adcb515e",
  "cutouts_relearn_l_v1.json": "d4d5fb8977b6ee231449816cbed9363357ce50a437389a260ec67082f86c205d"
 },
 "config_sha256": "2611d9bc8660c6c980fee47ec130d000da04733d52441c1e8fc37eafe680884e",
 "limi